In [48]:
# ------------------------------------------------------------
# 경고 제거
# ------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ============================================================
# 전자상거래 성과 분석 대시보드
# Google Colab 최종버전
# ============================================================

!apt-get -qq update
!apt-get -qq install fonts-nanum
!pip -q install openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from google.colab import files

# ============================================================
# 한글 폰트 설정
# ============================================================

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [57]:
# ==========================================================
# 한국은행 금통위 의사록 Hawkish / Dovish 분석
# Google Colab 실행용
#
# 기능
# 1. PDF 업로드
# 2. PDF 텍스트 추출
# 3. KoNLPy 형태소 분석
# 4. 상승(긴축) 시그널 분석
# 5. 하락(완화) 시그널 분석
# 6. 총 시그널 계산
# 7. 상승 시그널 비율 계산
# ==========================================================

# ----------------------------------------------------------
# 라이브러리 설치
# ----------------------------------------------------------

!apt-get -qq install openjdk-11-jdk-headless > /dev/null
!pip -q install konlpy
!pip -q install pdfplumber

# ----------------------------------------------------------
# 라이브러리
# ----------------------------------------------------------

import io
import pdfplumber

from google.colab import files
from konlpy.tag import Okt

# ----------------------------------------------------------
# PDF 업로드
# ----------------------------------------------------------

print("금통위 의사록 PDF를 업로드하세요.")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

# ----------------------------------------------------------
# PDF 텍스트 추출
# ----------------------------------------------------------

text = ""

with pdfplumber.open(
    io.BytesIO(uploaded[filename])
) as pdf:

    for page in pdf.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

print("텍스트 추출 완료")
print("총 글자수 :", format(len(text), ","))

# ----------------------------------------------------------
# 형태소 분석
# ----------------------------------------------------------

okt = Okt()

tokens = okt.morphs(
    text,
    stem=True
)

# ----------------------------------------------------------
# 상승(긴축) 키워드
# ----------------------------------------------------------

hawkish_keywords = [

    "상승",
    "인상",
    "확대",
    "증가",
    "강화",
    "회복",
    "상방",
    "높다",
    "오르다"
]

# ----------------------------------------------------------
# 하락(완화) 키워드
# ----------------------------------------------------------

dovish_keywords = [

    "하락",
    "위험",
    "축소",
    "감소",
    "약화",
    "둔화",
    "하방",
    "낮다",
    "내리다"
]

# ----------------------------------------------------------
# 빈도 계산
# ----------------------------------------------------------

hawkish_count = {}

for word in hawkish_keywords:

    hawkish_count[word] = tokens.count(word)

dovish_count = {}

for word in dovish_keywords:

    dovish_count[word] = tokens.count(word)

# ----------------------------------------------------------
# 총계
# ----------------------------------------------------------

hawkish_total = sum(hawkish_count.values())

dovish_total = sum(dovish_count.values())

total_signal = hawkish_total + dovish_total

if total_signal > 0:
    hawkish_ratio = hawkish_total / total_signal * 100
else:
    hawkish_ratio = 0

# ----------------------------------------------------------
# 결과 출력
# ----------------------------------------------------------

print("\n")
print("="*55)
print(f"분석 대상 : {filename}")
print("="*55)

print("\n[상승 및 강세 관련 키워드 등장 횟수]\n")

for word, cnt in sorted(
    hawkish_count.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(f"- {word:<6}: {cnt} 회")

print(f"\n▶ 상승 시그널 총계 : {hawkish_total:,} 회")

print("\n")
print("[하락 및 약세 관련 키워드 등장 횟수]\n")

for word, cnt in sorted(
    dovish_count.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(f"- {word:<6}: {cnt} 회")

print(f"\n▶ 하락 시그널 총계 : {dovish_total:,} 회")

print("\n")
print("="*55)
print("분석 요약")
print("="*55)

if hawkish_ratio >= 60:

    stance = "상승(긴축) 기조 우세"

elif hawkish_ratio <= 40:

    stance = "하락(완화) 기조 우세"

else:

    stance = "중립적 스탠스"

print(f"\n본 의사록은 {stance}를 보이고 있습니다.")

print(
    f"\n주요 시그널 점수 : {hawkish_ratio:.1f}%"
)

print(
    f"상승 시그널 비율 : {hawkish_ratio:.1f}%"
)

print(
    f"하락 시그널 비율 : {100-hawkish_ratio:.1f}%"
)

금통위 의사록 PDF를 업로드하세요.


Saving 2026년도 제7차 금통위 의사록.pdf to 2026년도 제7차 금통위 의사록 (1).pdf
텍스트 추출 완료
총 글자수 : 43,813


분석 대상 : 2026년도 제7차 금통위 의사록 (1).pdf

[상승 및 강세 관련 키워드 등장 횟수]

- 상승    : 115 회
- 확대    : 84 회
- 높다    : 58 회
- 증가    : 26 회
- 상방    : 22 회
- 회복    : 21 회
- 강화    : 19 회
- 인상    : 7 회
- 오르다   : 7 회

▶ 상승 시그널 총계 : 359 회


[하락 및 약세 관련 키워드 등장 횟수]

- 하방    : 27 회
- 둔화    : 23 회
- 위험    : 22 회
- 하락    : 20 회
- 약화    : 13 회
- 낮다    : 13 회
- 축소    : 9 회
- 감소    : 2 회
- 내리다   : 0 회

▶ 하락 시그널 총계 : 129 회


분석 요약

본 의사록은 상승(긴축) 기조 우세를 보이고 있습니다.

주요 시그널 점수 : 73.6%
상승 시그널 비율 : 73.6%
하락 시그널 비율 : 26.4%
